# Data Skew Optimization: Salting, Skew Hints, and AQE Skew Joins

**Module**: 10_performance_and_optimization  
**Notebook**: 06_data_skew_optimization.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Comprehensive hands-on laboratory for diagnosing and fixing data skew: key salting, broadcast join avoidance, and AQE automatic skew splitting.
- **Why does this topic exist?**: Data skew is the primary cause of jobs hanging at 99% progress.
- **What problem does it solve?**: 1 task running for 40 minutes while all other tasks finished in seconds.
- **Where is it used in real-world Data Engineering?**: Handling null keys and power-law customer purchase distributions.

---

## 2. Skew Remediation Playbook: 1. Broadcast Join -> 2. Salting -> 3. AQE Skew Join -> 4. Isolate Skewed Keys

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Skew_Opt_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Create skewed dataset
skewed = spark.range(0, 100000).withColumn("key", F.when(F.col("id") < 90000, "HOT_KEY").otherwise(F.concat(F.lit("KEY_"), F.col("id"))))
print("Key distribution:")
skewed.groupBy("key").count().orderBy(F.desc("count")).show(3)

---

## Experiment: Key Salting Implementation

- **1. What are we testing?**: Salt the skewed key to distribute the hotspot across 8 independent partitions.
- **2. What do we expect?**: The single massive partition is split into 8 uniform slices.

In [ ]:
salted = skewed.withColumn("salted_key", F.concat(F.col("key"), F.lit("_"), F.floor(F.rand() * 8)))
print("Salted key breakdown:")
salted.filter("key = 'HOT_KEY'").groupBy("salted_key").count().show()

**3. What happened?**: The 90,000 HOT_KEY rows were distributed across 8 salted keys (~11,250 rows each).  
**4. Why did it happen?**: Random salt assigned rows across distinct hash partitions.  
**5. Practical Engineering Lesson**: Salting converts an asymmetric bottleneck into uniform parallel task waves.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Skew Column
**Problem Statement**: Profile top keys in dirty dataset.  
**Input / Context**: dirty_df

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Filter Skewed Nulls
**Problem Statement**: Isolate null keys prior to join.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): AQE Skew Join Trigger
**Problem Statement**: Threshold settings for AQE skew join.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Skew Column):
```python
spark.read.option('header', True).csv('/opt/spark-data/quality/orders_dirty.csv').groupBy('customer_id').count().orderBy(F.desc('count')).show(5)
```

#### Solution 2 (Filter Skewed Nulls):
```python
print('Filter isNotNull() on left and right keys; union null records separately.')
```

#### Solution 3 (AQE Skew Join Trigger):
```python
print('spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes (default 256MB)')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Salting Formula**: key +'_'+ rand(0, K).
- **AQE Fix**: AQE automatically splits skewed shuffle partitions into sub-partitions.

### Essential Technical Terms
- `Data Skew`: Asymmetric record distribution.
- `Salting`: Adding random prefix to split key hotspots.

### Common Engineering Mistakes
- **Assuming more memory fixes skew**: If all rows map to 1 key, 1 task will still process them sequentially.

### Interview Questions on This Topic
- **Q: Explain how salting resolves a data skew join bottleneck.**
  - *A*: Salting appends a random number to the skewed key of the large table and replicates the matching small table keys with matching numbers, distributing the hotspot across multiple parallel partitions.

### What to Study Next
Open 07_performance_benchmarking.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')